# 2) Other ways to solve the power flow — Exercises

**Goals:** write the Gauss–Seidel update and test it at the solution, check what the FDLF matrix is made of, tune Gauss–Seidel on a larger network, compare the two FDLF variants, and measure what reusing a factorisation is worth over a time series.

The setup cell holds the lecture's `network(name, r_scale)` loader, `mismatch` and the sparse `newton_sparse` (which also accepts other injections `S` and a start `V0`). Everything is in per unit.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import scipy.sparse as sp
import scipy.sparse.linalg as spl
from pandapower.pypower.makeB import makeB

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)


def network(name, r_scale=1.0):
    """The lecture's loader: a solved pandapower case and its internal data, slack angle included."""
    net = getattr(pn, name)()
    net.line.r_ohm_per_km *= r_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)
    internal, order = net._ppc["internal"], net._pd2ppc_lookups["bus"]
    V0 = np.ones(len(internal["Sbus"]), dtype=complex)
    V0[order[net.gen.bus.to_numpy()]] = net.gen.vm_pu.to_numpy()
    V0[order[net.ext_grid.bus.to_numpy()]] = (net.ext_grid.vm_pu * np.exp(1j * np.radians(net.ext_grid.va_degree))).to_numpy()
    return {"net": net, "Y": internal["Ybus"].tocsr(), "S": internal["Sbus"].copy(), "V0": V0, "pv": internal["pv"],
            "pq": internal["pq"], "bus": internal["bus"], "branch": internal["branch"], "base": net._ppc["baseMVA"],
            "order": order, "V_pandapower": internal["V"]}


def mismatch(d, V, S=None):
    S = d["S"] if S is None else S
    m = V * np.conj(d["Y"] @ V) - S
    return max(np.abs(m[np.r_[d["pv"], d["pq"]]].real).max(), np.abs(m[d["pq"]].imag).max())


def newton_sparse(d, S=None, V0=None, tol=1e-8, max_iter=50):
    """The lecture's sparse Newton-Raphson (optionally with other injections S and start V0)."""
    Y, S = d["Y"], d["S"] if S is None else S
    V = (d["V0"] if V0 is None else V0).copy()
    pv, pq = d["pv"], d["pq"]
    pvpq = np.r_[pv, pq]
    for it in range(max_iter):
        I = Y @ V
        m = V * np.conj(I) - S
        F = np.r_[m[pvpq].real, m[pq].imag]
        if np.abs(F).max() < tol:
            return V, it
        dV, dI, dVn = sp.diags(V), sp.diags(I), sp.diags(V / np.abs(V))
        dS_dVa = 1j * dV @ np.conj(dI - Y @ dV)
        dS_dVm = dV @ np.conj(Y @ dVn) + np.conj(dI) @ dVn
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])]).tocsc()
        dx = spl.spsolve(J, -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("Newton: no convergence")

### Warm-ups

**1. The Gauss–Seidel update**

Write `gs_update(Y, V, S_i, i, is_pv, v_set)`: the new voltage of bus i from the lecture's formula (without acceleration) — for a PV bus, compute its reactive injection from the current voltages first and rescale the result to `v_set`. Test the defining property: at pandapower's solution, the update must return the bus's voltage unchanged, for every PV and PQ bus of the 14-bus system.

In [2]:
def gs_update(Y: np.ndarray, V: np.ndarray, S_i: complex, i: int, is_pv: bool, v_set: float) -> complex:
    """One Gauss-Seidel update of bus i."""
    ### BEGIN SOLUTION
    s = S_i
    if is_pv:
        s = S_i.real + 1j * (V[i] * np.conj(Y[i] @ V)).imag
    new = (np.conj(s / V[i]) - (Y[i] @ V - Y[i, i] * V[i])) / Y[i, i]
    return v_set * new / abs(new) if is_pv else new
    ### END SOLUTION

In [3]:
d14 = network("case14")
Y14, V_star = d14["Y"].toarray(), d14["V_pandapower"]
pv_set = set(d14["pv"].tolist())
changes = [abs(gs_update(Y14, V_star, d14["S"][i], i, i in pv_set, abs(V_star[i])) - V_star[i])
           for i in np.r_[d14["pv"], d14["pq"]]]
assert max(changes) < 1e-8, "the solution is a fixed point of the update"
print(f"largest change at the solution: {max(changes):.1e} p.u.")

largest change at the solution: 4.6e-16 p.u.


**2. What is B′ made of?**

Build B′ and B″ with `makeB(base, bus, branch, 2)` for the 14-bus system. Check that B′ is symmetric, and that for every line without an off-nominal tap the off-diagonal entry B′[i, k] equals −1/x of the line (x in p.u., from column 3 of `branch`; parallel lines add). Write `check_bprime(d)` returning the largest deviation.

In [4]:
def check_bprime(d) -> float:
    """Largest deviation between B'[i, k] and the sum of -1/x over the branches joining i and k (no taps)."""
    ### BEGIN SOLUTION
    B1, _ = makeB(d["base"], d["bus"], np.real(d["branch"]), 2)
    B1 = B1.toarray()
    assert np.allclose(B1, B1.T), "B' is symmetric"
    br = np.real(d["branch"])
    expected = {}
    for row in br:
        i, k, x, tap = int(row[0]), int(row[1]), row[3], row[8]
        if tap not in (0.0, 1.0):
            continue
        key = tuple(sorted((i, k)))
        expected[key] = expected.get(key, 0.0) - 1.0 / x
    return max(abs(B1[i, k] - value) for (i, k), value in expected.items())
    ### END SOLUTION

In [5]:
deviation = check_bprime(d14)
assert deviation < 1e-9
print(f"B' off-diagonals equal -1/x to {deviation:.1e}")

B' off-diagonals equal -1/x to 0.0e+00


### Core

**3. Gauss–Seidel on a larger network**

Implement `gauss_seidel(d, alpha, tol=1e-8, max_sweeps=20000)` with your `gs_update` (and the acceleration factor applied before the PV rescaling, as in the lecture). Run it on the **118-bus** system for α in [1.0, 1.4, 1.6, 1.8] and report the sweeps needed. Compare with the 14-bus results of the lecture (245 sweeps at α = 1.0, 51 at α = 1.6).

In [6]:
def gauss_seidel(d, alpha: float = 1.0, tol: float = 1e-8, max_sweeps: int = 20_000):
    """Gauss-Seidel power flow with acceleration. Returns V and the number of sweeps."""
    ### BEGIN SOLUTION
    Y = d["Y"].toarray()
    V = d["V0"].copy()
    pv = set(d["pv"].tolist())
    v_set = np.abs(d["V0"])
    for sweep in range(1, max_sweeps + 1):
        for i in np.r_[d["pv"], d["pq"]]:
            s = d["S"][i]
            if i in pv:
                s = s.real + 1j * (V[i] * np.conj(Y[i] @ V)).imag
            new = (np.conj(s / V[i]) - (Y[i] @ V - Y[i, i] * V[i])) / Y[i, i]
            V[i] = V[i] + alpha * (new - V[i])
            if i in pv:
                V[i] = v_set[i] * V[i] / abs(V[i])
        if mismatch(d, V) < tol:
            return V, sweep
    raise RuntimeError("no convergence")
    ### END SOLUTION

In [7]:
d118 = network("case118")
rows = {}
for alpha in [1.0, 1.4, 1.6, 1.8]:
    V, sweeps = gauss_seidel(d118, alpha)
    assert np.abs(V - d118["V_pandapower"]).max() < 1e-6
    rows[alpha] = sweeps
gs118 = pd.Series(rows, name="sweeps, 118 buses").rename_axis("alpha")
assert gs118.min() > 5 * 51, "far more sweeps than on 14 buses"
gs118

alpha
1.0    2794
1.4    1267
1.6     777
1.8     378
Name: sweeps, 118 buses, dtype: int64

**4. Two FDLF variants**

`makeB` builds B′ and B″ in two standard variants: **XB** (`alg=2`, the lecture's: resistance ignored in B′) and **BX** (`alg=3`: resistance ignored in B″ instead). Write `fdlf(d, variant)` and compare the iterations of both on the 14-bus system with line resistance scaled ×1, ×2, ×3 and ×4.

In [8]:
def fdlf(d, variant: int = 2, S=None, V0=None, tol=1e-8, max_iter=500, factors=None):
    """Fast decoupled load flow (variant 2 = XB, 3 = BX). Returns V, iterations and the factorisations."""
    ### BEGIN SOLUTION
    S = d["S"] if S is None else S
    pv, pq = d["pv"], d["pq"]
    pvpq = np.r_[pv, pq]
    if factors is None:
        B1, B2 = makeB(d["base"], d["bus"], np.real(d["branch"]), variant)
        factors = (spl.splu(B1[pvpq][:, pvpq].tocsc()), spl.splu(B2[pq][:, pq].tocsc()))
    lu1, lu2 = factors
    V = (d["V0"] if V0 is None else V0).copy()
    for it in range(max_iter):
        if mismatch(d, V, S) < tol:
            return V, it, factors
        Va, Vm = np.angle(V), np.abs(V)
        m = V * np.conj(d["Y"] @ V) - S
        Va[pvpq] -= lu1.solve(m[pvpq].real / Vm[pvpq])
        V = Vm * np.exp(1j * Va)
        m = V * np.conj(d["Y"] @ V) - S
        Vm[pq] -= lu2.solve(m[pq].imag / Vm[pq])
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("FDLF: no convergence")
    ### END SOLUTION

In [9]:
rows = {}
for r_scale in [1, 2, 3, 4]:
    d = network("case14", r_scale)
    rows[r_scale] = {"XB": fdlf(d, 2)[1], "BX": fdlf(d, 3)[1], "Newton": newton_sparse(d)[1]}
variants = pd.DataFrame(rows).T.rename_axis("line resistance x")
assert variants.loc[1, "XB"] == 8, "the lecture's 8 iterations"
variants

                   XB  BX  Newton
line resistance x                
1                   8  10       4
2                  18  12       4
3                  34  13       4
4                  83  28       5

**5. Interpretation**

How does Gauss–Seidel's sweep count scale from 14 to 118 buses, and does the best acceleration factor carry over? What did you find about the two FDLF variants as resistance grows — and does either one remove the weakness?

<!-- BEGIN SOLUTION -->
Gauss–Seidel scales badly. Without acceleration it needs 2,794 sweeps on 118 buses against 245 on 14 — eleven times as many sweeps on a network eight times larger — and each of those sweeps updates eight times as many buses. Acceleration still helps, but the best factor **does not carry over**: on 14 buses α = 1.6 was best and 1.8 overshot, while on 118 buses α = 1.8 is the best of those tried (378 sweeps). The optimum depends on the network and has to be tuned, and even tuned the method needs hundreds of sweeps where Newton needs four — which is why Gauss–Seidel survives mainly as a teaching method and as a cheap way to improve a starting point.

The two FDLF variants trade places as resistance grows. On the original network XB is slightly faster (8 against 10 iterations); with line resistance doubled BX already wins (12 against 18), and at four times the resistance BX needs 28 iterations where XB needs 83. The variants differ in where they drop the resistance — XB leaves it out of B′, BX out of B″ — and on this network keeping it in B′ pays off once resistance is no longer small. But neither removes the weakness: at four times the resistance even BX needs 28 iterations against Newton's 5, because both freeze matrices that ignore part of the coupling.
<!-- END SOLUTION -->

### Challenge

**6. A time series: factorise once, solve many**

Solve **200 load snapshots** of the 1,354-bus system (`case1354pegase`): in snapshot t every load is scaled by a factor drawn uniformly from 0.9–1.1 (seed 0), generators' active power unchanged — so S_t = S_gen − factor_t · S_load. Compare two strategies: (a) `newton_sparse` for each snapshot, starting from the previous snapshot's solution; (b) `fdlf` with B′ and B″ **factorised once** and reused for all 200 snapshots, also warm-started. Report total time, total iterations and the largest difference between the two strategies' voltages. Write `time_series(n=200)`. (Not every test case suits this: the 300-bus case, with its generators fixed, has no solution already at 0.9 or 1.05 times its base load — pandapower's solvers fail there too.)

In [10]:
def time_series(n: int = 200) -> dict:
    """Total time (s) and iterations of Newton and of FDLF with one factorisation over n load snapshots."""
    ### BEGIN SOLUTION
    d = network("case1354pegase")
    net = d["net"]
    S_load = np.zeros(len(d["S"]), dtype=complex)
    np.add.at(S_load, d["order"][net.load.bus.to_numpy()],
              (net.load.p_mw.to_numpy() + 1j * net.load.q_mvar.to_numpy()) / d["base"])
    S_gen = d["S"] + S_load
    factors_t = np.random.default_rng(0).uniform(0.9, 1.1, n)
    out = {}
    for label in ["Newton", "FDLF, one factorisation"]:
        V, iterations, factors, solutions = d["V0"].copy(), 0, None, []
        start = time.perf_counter()
        for f in factors_t:
            S = S_gen - f * S_load
            if label == "Newton":
                V, it = newton_sparse(d, S=S, V0=V)
            else:
                V, it, factors = fdlf(d, 2, S=S, V0=V, factors=factors)
            iterations += it
            solutions.append(V)
        out[label] = {"seconds": time.perf_counter() - start, "iterations": iterations, "solutions": np.array(solutions)}
    out["largest_difference"] = float(np.abs(out["Newton"]["solutions"] - out["FDLF, one factorisation"]["solutions"]).max())
    return out
    ### END SOLUTION

In [11]:
ts = time_series()
assert ts["largest_difference"] < 1e-6, "both strategies solve the same 200 power flows"
summary = pd.DataFrame({k: {"seconds": v["seconds"], "iterations": v["iterations"]} for k, v in ts.items() if isinstance(v, dict)}).T
print(f"largest difference between the strategies: {ts['largest_difference']:.1e} p.u. (timings depend on the machine)")
summary.round(2)

largest difference between the strategies: 4.7e-10 p.u. (timings depend on the machine)


                         seconds  iterations
Newton                      6.20       741.0
FDLF, one factorisation     0.81      2082.0

**7. Interpretation**

How much faster is FDLF over the time series, and why is the advantage larger than in the lecture's single power flows? Name a real study where this pattern matters.

<!-- BEGIN SOLUTION -->
Over the 200 snapshots FDLF with one factorisation was about **six to eight times faster** than Newton in the runs that built this key (see the table), although it needed nearly three times as many iterations (2,082 against 741) — so each FDLF iteration cost roughly a twentieth of a Newton iteration. The advantage is larger than in the lecture's single power flows (two to three times) because the expensive step — factorising B′ and B″ — is done once for all 200 snapshots, while Newton builds and factorises a new Jacobian in every one of its 741 iterations. Warm starts help both methods: each snapshot starts from a solution only a few per cent away. The exact seconds vary between machines, the ratio much less.

This pattern matters wherever the same network is solved many times with different injections: an annual time series of 8,760 hourly power flows, Monte Carlo studies of renewable or load uncertainty, or real-time operation where the network is re-solved every few minutes. In N-1 contingency screening the network itself changes with every outage, so B′ and B″ change too — there, engineers update the existing factorisation for the one removed branch instead of refactorising, which keeps most of the saving (lesson 6).
<!-- END SOLUTION -->